# Operating on Data in Pandas

One of the strengths of NumPy is that it allows us to perform quick element-wise operations, both with basic arithmetic (addition, subtraction, multiplication, etc.) and with more complicated operations (trigonometric functions, exponential and logarithmic functions, etc.).
Pandas inherits much of this functionality from NumPy, and the ufuncs introduced in [Computation on NumPy Arrays: Universal Functions](02.03-Computation-on-arrays-ufuncs.ipynb) are key to this.

Pandas includes a couple of useful twists, however: for unary operations like negation and trigonometric functions, these ufuncs will *preserve index and column labels* in the output, and for binary operations such as addition and multiplication, Pandas will automatically *align indices* when passing the objects to the ufunc.
This means that keeping the context of data and combining data from different sources—both potentially error-prone tasks with raw NumPy arrays—become essentially foolproof with Pandas.
We will additionally see that there are well-defined operations between one-dimensional `Series` structures and two-dimensional `DataFrame` structures.

## Ufuncs: Index Preservation

Because Pandas is designed to work with NumPy, any NumPy ufunc will work on Pandas `Series` and `DataFrame` objects.
Let's start by defining a simple `Series` and `DataFrame` on which to demonstrate this:

In [3]:
import pandas as pd
import numpy as np

In [5]:
np.random.seed(42)
ser = pd.Series(np.random.randint(1, 10, 4))
ser

0    7
1    4
2    8
3    5
dtype: int32

In [6]:
df = pd.DataFrame(np.random.randint(0, 10, (3, 4)), 
                  columns=['A', 'B', 'C', 'D']  )
df

,A,B,C,D
0,6,9,2,6
1,7,4,3,7
2,7,2,5,4


If we apply a NumPy ufunc on either of these objects, the result will be another Pandas object *with the indices preserved:*

In [7]:
np.exp(ser)

0    1096.633158
1      54.598150
2    2980.957987
3     148.413159
dtype: float64

In [8]:
df

,A,B,C,D
0,6,9,2,6
1,7,4,3,7
2,7,2,5,4


In [9]:
np.sin(df * np.pi / 4)

,A,B,C,D
0,-1.000000,7.071068e-01,1.000000,-1.000000e+00
1,-0.707107,1.224647e-16,0.707107,-7.071068e-01
2,-0.707107,1.000000e+00,-0.707107,1.224647e-16


This is true also for more involved sequences of operations:

In [33]:
np.sin(df * np.pi / 4)

,A,B,C,D
0,-1.000000,7.071068e-01,1.000000,-1.000000e+00
1,-0.707107,1.224647e-16,0.707107,-7.071068e-01
2,-0.707107,1.000000e+00,-0.707107,1.224647e-16


Any of the ufuncs discussed in [Computation on NumPy Arrays: Universal Functions](02.03-Computation-on-arrays-ufuncs.ipynb) can be used in a similar manner.

## Ufuncs: Index Alignment

For binary operations on two `Series` or `DataFrame` objects, Pandas will align indices in the process of performing the operation.
This is very convenient when working with incomplete data, as we'll see in some of the examples that follow.

### Index Alignment in Series

As an example, suppose we are combining two different data sources and wish to find only the top three US states by *area* and the top three US states by *population*:

In [10]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 39538223, 'Texas': 29145505,
                        'Florida': 21538187}, name='population')

In [11]:
area

Alaska        1723337
Texas          695662
California     423967
Name: area, dtype: int64

In [12]:
population

California    39538223
Texas         29145505
Florida       21538187
Name: population, dtype: int64

In [14]:
population/area

Alaska              NaN
California    93.257784
Florida             NaN
Texas         41.896072
dtype: float64

In [15]:
area.index.union(population.index)

Index(['Alaska', 'California', 'Florida', 'Texas'], dtype='str')

In [35]:
print(area)
print(population)

Alaska        1723337
Texas          695662
California     423967
Name: area, dtype: int64
California    39538223
Texas         29145505
Florida       21538187
Name: population, dtype: int64


Let's see what happens when we divide these to compute the population density:

In [12]:
population / area

Alaska              NaN
California    93.257784
Florida             NaN
Texas         41.896072
dtype: float64

The resulting array contains the *union* of indices of the two input arrays, which could be determined directly from these indices:

In [15]:
area.index.union(population.index)

Index(['Alaska', 'California', 'Florida', 'Texas'], dtype='str')

Any item for which one or the other does not have an entry is marked with `NaN`, or "Not a Number," which is how Pandas marks missing data (see further discussion of missing data in [Handling Missing Data](03.04-Missing-Values.ipynb)).
This index matching is implemented this way for any of Python's built-in arithmetic expressions; any missing values are marked by `NaN`:

In [16]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
print(A)
print(B)

0    2
1    4
2    6
dtype: int64
1    1
2    3
3    5
dtype: int64


In [17]:
A + B

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

In [18]:
A.add(B, fill_value = 0 )

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

In [20]:
A  = pd.DataFrame(np.random.randint(1, 20, (2, 2)),
                   columns=['A', 'B'])
A

,A,B
0,2,12
1,6,2


In [ ]:
B = pd.DataFrame(np.random.randint(0, 10, (3, 3)),
                                columns=['A', 'B', 'C'])
B

,A,B,C
0,4,0,9
1,5,8,0
2,9,2,6


In [22]:
A + B

,A,B,C
0,6.0,12.0,NaN
1,11.0,10.0,NaN
2,NaN,NaN,NaN


In [23]:
A.add(B, fill_value = A.values.mean())

,A,B,C
0,6.0,12.0,14.5
1,11.0,10.0,5.5
2,14.5,7.5,11.5


In [40]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
A + B

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

If using `NaN` values is not the desired behavior, the fill value can be modified using appropriate object methods in place of the operators.
For example, calling ``A.add(B)`` is equivalent to calling ``A + B``, but allows optional explicit specification of the fill value for any elements in ``A`` or ``B`` that might be missing:

In [39]:
print(A)
print(B)

0    2
1    4
2    6
dtype: int64
1    1
2    3
3    5
dtype: int64


In [12]:
A.add(B, fill_value = 0 )

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

### Index Alignment in DataFrames

A similar type of alignment takes place for *both* columns and indices when performing operations on `DataFrame` objects:

In [18]:
A = pd.DataFrame(np.random.randint(0, 20, (2, 2)),
                 columns =  ['a', 'b'])
A

,a,b
0,11,19
1,2,4


In [19]:
B = pd.DataFrame(np.random.randint(0, 10, (3, 3)),
                 columns = ['b', 'c', 'd'])
B


,b,c,d
0,2,6,4
1,8,6,1
2,3,8,1


In [20]:
A + B

,a,b,c,d
0,NaN,21.0,NaN,NaN
1,NaN,12.0,NaN,NaN
2,NaN,NaN,NaN,NaN


In [21]:
A.add(B, fill_value = A.values.mean())

,a,b,c,d
0,20.0,21.0,15.0,13.0
1,11.0,12.0,15.0,10.0
2,NaN,12.0,17.0,10.0


In [17]:
A.add(B, fill_value = A.values.mean())

,a,b,c,d
0,5.5,15.0,4.5,13.5
1,9.5,6.0,12.5,4.5
2,NaN,13.5,6.5,10.5


In [22]:
B = pd.DataFrame(np.random.randint(0, 10, (3, 3)),
                 columns=['b', 'a', 'c'])
B

,b,a,c
0,9,8,9
1,4,1,3
2,6,7,2


In [23]:
A + B

,a,b,c
0,19.0,28.0,NaN
1,3.0,8.0,NaN
2,NaN,NaN,NaN


Notice that indices are aligned correctly irrespective of their order in the two objects, and indices in the result are sorted.
As was the case with `Series`, we can use the associated object's arithmetic methods and pass any desired `fill_value` to be used in place of missing entries.
Here we'll fill with the mean of all values in `A`:

In [24]:
A.add(B, fill_value=A.values.mean())

,a,b,c
0,19.0,28.0,18.0
1,3.0,8.0,12.0
2,16.0,15.0,11.0


The following table lists Python operators and their equivalent Pandas object methods:

| Python operator | Pandas method(s)                |
|-----------------|---------------------------------|
| `+`             | `add`                           |
| `-`             | `sub`, `subtract`               |
| `*`             | `mul`, `multiply`               |
| `/`             | `truediv`, `div`, `divide`      |
| `//`            | `floordiv`                      |
| `%`             | `mod`                           |
| `**`            | `pow`                           |


## Ufuncs: Operations Between DataFrames and Series

When performing operations between a `DataFrame` and a `Series`, the index and column alignment is similarly maintained, and the result is similar to operations between a two-dimensional and one-dimensional NumPy array.
Consider one common operation, where we find the difference of a two-dimensional array and one of its rows:

In [25]:
np.random.seed(42)
A = np.random.randint(10, size=(3, 4))
A

array([[6, 3, 7, 4],
       [6, 9, 2, 6],
       [7, 4, 3, 7]], dtype=int32)

In [27]:
A - A[0]

array([[ 0,  0,  0,  0],
       [ 0,  6, -5,  2],
       [ 1,  1, -4,  3]], dtype=int32)

In [29]:
df = pd.DataFrame(A, columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
0,6,3,7,4
1,6,9,2,6
2,7,4,3,7


In [30]:
df.subtract(df['A'], axis = 0)

,A,B,C,D
0,0,-3,1,-2
1,0,3,-4,0
2,0,-3,-4,0


In [25]:
np.random.seed(42)
A = np.random.randint(10, size=(3, 4))
A

array([[6, 3, 7, 4],
       [6, 9, 2, 6],
       [7, 4, 3, 7]], dtype=int32)

In [20]:
A - A[0]

array([[ 0,  0,  0,  0],
       [ 0,  6, -5,  2],
       [ 1,  1, -4,  3]], dtype=int32)

In [26]:
df = pd.DataFrame(A, columns = ['A', 'B', 'C', 'D'])
df

,A,B,C,D
0,6,3,7,4
1,6,9,2,6
2,7,4,3,7


In [27]:
df.iloc[0]

A    6
B    3
C    7
D    4
Name: 0, dtype: int32

In [28]:
df - df.iloc[0]

,A,B,C,D
0,0,0,0,0
1,0,6,-5,2
2,1,1,-4,3


In [27]:
df.subtract(df['B'], axis = 0)

,A,B,C,D
0,3,0,4,1
1,-3,0,-7,-3
2,3,0,-1,3


In [29]:
df

,A,B,C,D
0,6,3,7,4
1,6,9,2,6
2,7,4,3,7


In [32]:
halfrow = df.iloc[0, [0,2]]
halfrow

A    6
C    7
Name: 0, dtype: int32

In [33]:
df - halfrow

,A,B,C,D
0,0.0,NaN,0.0,NaN
1,0.0,NaN,-5.0,NaN
2,1.0,NaN,-4.0,NaN


In [23]:
np.random.seed(42)
A = np.random.randint(10, size=(3, 4))
A

array([[6, 3, 7, 4],
       [6, 9, 2, 6],
       [7, 4, 3, 7]], dtype=int32)

In [24]:
A - A[0]

array([[ 0,  0,  0,  0],
       [ 0,  6, -5,  2],
       [ 1,  1, -4,  3]], dtype=int32)

According to NumPy's broadcasting rules (see [Computation on Arrays: Broadcasting](02.05-Computation-on-arrays-broadcasting.ipynb)), subtraction between a two-dimensional array and one of its rows is applied row-wise.

In Pandas, the convention similarly operates row-wise by default:

In [25]:
df = pd.DataFrame(A, columns=['Q', 'R', 'S', 'T'])
df - df.iloc[0]

,Q,R,S,T
0,0,0,0,0
1,0,6,-5,2
2,1,1,-4,3


If you would instead like to operate column-wise, you can use the object methods mentioned earlier, while specifying the `axis` keyword:

In [26]:
df.subtract(df['R'], axis=0)

,Q,R,S,T
0,3,0,4,1
1,-3,0,-7,-3
2,3,0,-1,3


Note that these `DataFrame`/`Series` operations, like the operations discussed previously, will automatically align  indices between the two elements:

In [28]:
df

,Q,R,S,T
0,6,3,7,4
1,6,9,2,6
2,7,4,3,7


In [27]:
halfrow = df.iloc[0, ::2]
halfrow

Q    6
S    7
Name: 0, dtype: int32

In [29]:
df - halfrow

,Q,R,S,T
0,0.0,NaN,0.0,NaN
1,0.0,NaN,-5.0,NaN
2,1.0,NaN,-4.0,NaN


This preservation and alignment of indices and columns means that operations on data in Pandas will always maintain the data context, which prevents the common errors that might arise when working with heterogeneous and/or misaligned data in raw NumPy arrays.